# **Modelo Híbrido**
### Proyecto Hito 1
### Sistemas Recomendadores IIC3633-1 2025-2
### **Grupo 3:** 

- Nicolás Antonio Bueno Abett de la Torre 

- Felipe Andrés Fuentes González

- Jorge Andrés Jacque Palma

- Francisco Nicolás Solís Gormaz

## Índice

>[0- Instalación de librerías](#0--instalación-de-librerías)

>[1- Carga de datos](#1--carga-de-datos)

>[2- User KNN](#2--user-knn)

>[3- Item KNN](#3--item-knn)

>[4- Most Popular](#4--most-popular)

>[5- Random](#5--random)

## 0- Instalación de librerías

In [61]:
# !pip uninstall -y numpy
# !pip install numpy==1.26

In [62]:
# !pip install scikit-surprise --no-build-isolation --no-deps

In [63]:
# pip install pandas

## 1- Carga de datos

In [64]:
import surprise
import numpy as np
import pandas as pd
from collections import defaultdict
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy
import random

Se lee el archivo de datos y se almacena en un dataframe:

In [65]:
df = pd.read_csv('video_game_reviews.csv')

Se añade la ID de usuario para cada evaluación del dataframe, ya que no poseen. Para ello, se generan 3000 ID de usuarios (del 1 al 3000) que se reparten aleatoriamente por las evaluaciones realizadas. Además, se fija la semilla del random para que sea replicable. Se agrega la columna "user_id" al dataframe:

In [66]:
np.random.seed(42)
df['user_id'] = np.random.randint(1, 3001, size=len(df))

Se añade ID de ítem, ya que tampoco se posee. Para eso, se asigna arbitrariamente una ID a cada videojuego que aparece en el set de datos. Ya que hay 40 videojuegos diferentes, se asignan ID desde el 1 al 40:

In [67]:
print(f'Videojuegos unicos: {df["Game Title"].unique()}')

Videojuegos unicos: ['Grand Theft Auto V' 'The Sims 4' 'Minecraft' 'Bioshock Infinite'
 'Half-Life: Alyx' 'Sid Meier’s Civilization VI' 'Just Dance 2024'
 '1000-Piece Puzzle' 'Spelunky 2' 'Street Fighter V' 'Fall Guys'
 'Rocket League' 'The Elder Scrolls V: Skyrim' 'Among Us' 'Stardew Valley'
 'Call of Duty: Modern Warfare 2'
 'The Legend of Zelda: Breath of the Wild' 'Tekken 7'
 'Pillars of Eternity II: Deadfire' 'Animal Crossing: New Horizons'
 'Hades' 'Mario Kart 8 Deluxe' 'Overwatch 2' 'Fortnite'
 'Pokémon Scarlet & Violet' 'Hitman 3' 'Tomb Raider (2013)'
 'Halo Infinite' 'Super Smash Bros. Ultimate' 'Kingdom Hearts III'
 'League of Legends' 'The Witcher 3: Wild Hunt' 'FIFA 24'
 'Ghost of Tsushima' 'Cuphead' 'Red Dead Redemption 2' 'Portal 2' 'Tetris'
 'Counter-Strike: Global Offensive' 'Super Mario Odyssey']


Se añade la columna "item_id" al dataframe:

In [68]:
df['item_id'] = df['Game Title'].astype('category').cat.codes + 1

Se define un diccionario para almacenar el ttítulo del videojuego a partir de su ID, para análisis posterior:

In [69]:
info_videojuegos = dict(zip(df['item_id'], df['Game Title']))

Reordenar columnas:

In [70]:
cols = df.columns.tolist()
new_order = ['user_id', 'item_id'] + [c for c in cols if c not in ['user_id', 'item_id']]
df = df[new_order]

Guardar dataframe reordenado y con las ID de usuario e ítem agregadas en un nuevo archivo .csv:

In [71]:
df.to_csv('video_game_reviews_with_userid.csv', index=False)

Reviews por usuario:

In [72]:
print(df['user_id'].value_counts())

user_id
2839    32
2135    30
948     30
2774    30
993     29
        ..
544      5
1581     5
2050     5
2465     5
424      5
Name: count, Length: 3000, dtype: int64


Verificar cantidad de usuarios:

In [73]:
print(f"Numero de usuarios unicos: {df['user_id'].nunique()}")

Numero de usuarios unicos: 3000


Se lee el último .csv y se guardan solamente las columnas de ID de usuario, ID de ítem, y el rating correspondiente:

In [74]:
df = pd.read_csv('video_game_reviews_with_userid.csv', sep=',')
df = df[['user_id', 'item_id', 'User Rating']]

Convertir ratings a escala del 1 al 5:

In [ ]:
def parametrizar_rating_a_5(df):
    vals = df["User Rating"].to_numpy().astype(float)

    minimo = np.min(vals)
    maximo = np.max(vals)

    scaled = 1 + ( (vals - minimo) / (maximo - minimo) ) * (5 - 1)

    df["rating"] = scaled

    return df

Se usa la función en el dataframe para generar la nueva columna "rating", con el valor del rating reescalado del 1 al 5, y se elimina la antigua columna "User Rating":

In [76]:
df = parametrizar_rating_a_5(df)
df.drop("User Rating", axis=1, inplace=True)

El dataframe final que se utilizará es:

In [77]:
df

,user_id,item_id,rating
0,861,12,3.670051
1,1295,38,3.862944
2,1131,21,2.695431
3,1096,4,3.873096
4,1639,14,3.030457
...,...,...,...
47769,1293,21,4.197970
47770,2485,37,2.431472
47771,2675,3,2.685279
47772,2599,37,2.258883


Se guarda el dataframe modificado en un nuevo archivo .csv:

In [78]:
df.to_csv('video_game_reviews_with_userid_clean.csv', index=False)

Se lee el .csv con los datos modificados y se definen los datasets de entrenamiento y testeo:

In [79]:
reader = Reader(line_format='user item rating', sep=',', rating_scale=(1,5), skip_lines=1)
data = Dataset.load_from_file('video_game_reviews_with_userid_clean.csv', reader=reader)

trainset, testset = train_test_split(data, test_size=0.2)

print("Usuarios:", trainset.n_users, "Items:", trainset.n_items, "Test size:", len(testset))

Usuarios: 3000 Items: 40 Test size: 9555


## 2- User KNN

Se busca el mejor valor de 'k' entre un conjunto predefinido 'k_values'. Se prueba usando la correlación de Pearson y la similitud de coseno como medidas de similtud. Finalmente, se elige el valor de 'k' y la métrica de similitud con los que se obtiene menor valor de RMSE:

In [ ]:
# código de Felipe Fuentes y Nicolás Bueno, usado en su tarea

#Valores de k que se probarán:
k_values = [5, 10, 20, 30, 50, 70, 100]
rmse_values_user_knn = []
sim_options = ["cosine", "pearson"]

for option in sim_options:
  for k in k_values:
    myUserKnn = surprise.KNNBasic(k=k, sim_options={'name': option, 'user_based': True})
    myUserKnn.fit(trainset)
    predictions = myUserKnn.test(testset)
    rmse_values_user_knn.append([option, k, accuracy.rmse(predictions, verbose=False)])

best_cosine = None
best_pearson = None

for sim, k, rmse in rmse_values_user_knn:
    if sim == "cosine":
        if best_cosine is None or rmse < best_cosine[2]:
            best_cosine = [sim, k, rmse]
    elif sim == "pearson":
        if best_pearson is None or rmse < best_pearson[2]:
            best_pearson = [sim, k, rmse]

print("Mejor cosine :", best_cosine)
print("Mejor pearson:", best_pearson)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Comput

Para realizar la recomendación, se define el modelo User KNN con similitud de coseno y k = 100, que es la combinación que alcanzó menor RMSE:

In [102]:
myUserKnn = surprise.KNNBasic(k=100, sim_options={'name': 'cosine', 'user_based': True})

In [103]:
myUserKnn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


Como ejemplo, se prueba predecir el rating que el usuario de ID = 457 daría al videojuego de ID = 1:

In [104]:
myUserKnn.predict("457", "1")

Prediction(uid='457', iid='1', r_ui=None, est=3.0522570155596944, details={'actual_k': 100, 'was_impossible': False})

Se realizan las predicciones a partir del antitest set:

In [105]:
a_testset = trainset.build_anti_testset()
predictions = myUserKnn.test(a_testset)

Función para obtener lista de recomendación top N:

In [106]:
def get_top_n(predictions, n=10):
    """Devuelve las N-mejores recomendaciones para cada usuario de un set de predicción.

    Args:
        predictions(lista de objetos Prediction): La lista de predicción obtenida del método test.
        n(int): El número de recomendaciónes por usuario

    Returns:
    Un diccionario donde las llaves son ids de usuario y los valores son listas de tuplas:
        [(item id, rating estimation), ...] de tamaño n.
    """

    # First map the predictions to each user.
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))

    # Then sort the predictions for each user and retrieve the k highest ones.
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]

    return top_n

Se obtiene la lista de recomendación top 10 para cada usuario y s emuestra como ejemplo el caso del con ID = 457:

In [107]:
top_n = get_top_n(predictions, n=10)
print(top_n["457"])

[('22', 3.1220690346885007), ('8', 3.103378061489484), ('29', 3.081722512851395), ('9', 3.070300424603499), ('1', 3.0522570155596944), ('7', 3.048995738315107), ('33', 3.0410856990498716), ('19', 3.031334445878307), ('10', 3.030264894240623), ('17', 3.004619618715724)]


Se muestran los títulos de los videojuegos recomendados al usuario ID = 457 junto a su rating predicho:

In [108]:
for item_id, rating in top_n["457"]:
    print(f"{info_videojuegos[int(item_id)]}: {rating:.2f}")

Overwatch 2: 3.12
FIFA 24: 3.10
Spelunky 2: 3.08
Fall Guys: 3.07
1000-Piece Puzzle: 3.05
Cuphead: 3.05
Super Smash Bros. Ultimate: 3.04
League of Legends: 3.03
Fortnite: 3.03
Just Dance 2024: 3.00


## 3- Item KNN

Se busca el mejor valor de K entre un conjunto predefinido 'k_values'. Se prueba usando la correlación de Pearson y la similitud de coseno como medidas de similtud. Finalmente, se elige el valor de K y la métrica de similitud con los que se obtiene menor valor de RMSE:

In [88]:
#Valores de k que se probarán:
k_values = [5, 10, 20, 30, 50, 70, 100]
rmse_values_user_knn = []
sim_options = ["cosine", "pearson"]

for option in sim_options:
  for k in k_values:
    myUserKnn = surprise.KNNBasic(k=k, sim_options={'name': option, 'user_based': False})
    myUserKnn.fit(trainset)
    predictions = myUserKnn.test(testset)
    rmse_values_user_knn.append([option, k, accuracy.rmse(predictions, verbose=False)])

best_cosine = None
best_pearson = None

for sim, k, rmse in rmse_values_user_knn:
    if sim == "cosine":
        if best_cosine is None or rmse < best_cosine[2]:
            best_cosine = [sim, k, rmse]
    elif sim == "pearson":
        if best_pearson is None or rmse < best_pearson[2]:
            best_pearson = [sim, k, rmse]

print("Mejor cosine :", best_cosine)
print("Mejor pearson:", best_pearson)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Comput

Para realizar la recomendación, se define el modelo Item KNN con similitud de coseno y k = 30, que es la combinación que alcanzó menor RMSE:

In [89]:
myItemKnn = surprise.KNNBasic(k=30, sim_options={'name': 'cosine', 'user_based': False})

In [90]:
myItemKnn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


Como ejemplo, se prueba predecir el rating que el usuario de ID = 457 daría al videojuego de ID = 1:

In [91]:
myItemKnn.predict("457", "0")

Prediction(uid='457', iid='0', r_ui=None, est=2.9902567928381756, details={'was_impossible': True, 'reason': 'User and/or item is unknown.'})

Se realizan las predicciones a partir del antitest set definido en la sección anterior:

In [92]:
predictions_item = myItemKnn.test(a_testset)

Se obtiene la lista de recomendación top 10 para cada usuario (usando get_top_n() definida arriba) y se muestra como ejemplo el caso del con ID = 457:

In [93]:
top_n_item = get_top_n(predictions_item, n=10)
print(top_n_item["457"])

[('1', 2.7297189862608087), ('4', 2.729477010553988), ('40', 2.729281703539494), ('10', 2.7292245767778227), ('32', 2.729149163894074), ('39', 2.729042902793737), ('28', 2.7289956726092583), ('17', 2.7289402499226796), ('2', 2.728918086852929), ('19', 2.728866083655035)]


Se muestran los títulos de los videojuegos recomendados al usuario ID = 457 junto a su rating predicho:

In [94]:
for item_id, rating in top_n_item["457"]:
    print(f"{info_videojuegos[int(item_id)]}: {rating:.8f}")

1000-Piece Puzzle: 2.72971899
Bioshock Infinite: 2.72947701
Tomb Raider (2013): 2.72928170
Fortnite: 2.72922458
Super Mario Odyssey: 2.72914916
The Witcher 3: Wild Hunt: 2.72904290
Sid Meier’s Civilization VI: 2.72899567
Just Dance 2024: 2.72894025
Among Us: 2.72891809
League of Legends: 2.72886608


## 4- Most Popular

Se realiza el procedimiento para recomendar a cada usuario los 10 videojuegos más populares que no haya visto. Para ello, se calcula para cada videojuego un 'score' de popularidad, que es la suma de todos los rating con el que los usuarios le han evaluado. Luego, los videojuegos más populares serán aquellos con el score más alto. Se define una función que realiza la recomendación top N para cada usuario:

In [ ]:
def get_top_n_most_popular(trainset, n=10):

    # se calcula el score de popularidad de cada item en el conjunto de entrenamiento
    item_popularity = defaultdict(int)
    for uid, iid, rating in trainset.all_ratings():
        item_popularity[iid] += rating

    # se ordenan los items por popularidad
    popular_items = sorted(item_popularity.items(), key=lambda x: x[1], reverse=True)

    # diccionario para las recomendaciones
    top_n = defaultdict(list)

    # para cada usuario, se recomiendan los n items con mayor socre de popularidad y que no haya visto
    for uid in trainset.all_users():
        user_items = set(iid for (iid, _) in trainset.ur[uid])
        count = 0
        for iid, _ in popular_items:
            if iid not in user_items:
                top_n[trainset.to_raw_uid(uid)].append((trainset.to_raw_iid(iid), item_popularity[iid]))
                count += 1
                if count >= n:
                    break

    return top_n

Se obtiene la lista de recomendación top 10 para cada usuario y se muestra como ejemplo el caso del con ID = 457. En la lista de recomendación, el primer elemento de cada tupla es la ID del ítem, y el segundo es su score de popularidad:

In [96]:
top_n_most_popular = get_top_n_most_popular(trainset, n=10)
print(top_n_most_popular["457"])

[('29', 2982.142131979695), ('8', 2973.3959390862947), ('40', 2952.741116751264), ('33', 2943.5837563451837), ('1', 2937.157360406091), ('22', 2899.5482233502535), ('39', 2896.837563451778), ('9', 2892.7258883248746), ('3', 2884.69543147208), ('23', 2860.893401015228)]


Se muestran los títulos de los videojuegos recomendados al usuario ID = 457 junto a su score de popularidad:

In [97]:
for item_id, score in top_n_most_popular["457"]:
    print(f"{info_videojuegos[int(item_id)]}: {score:.8f}")

Spelunky 2: 2982.14213198
FIFA 24: 2973.39593909
Tomb Raider (2013): 2952.74111675
Super Smash Bros. Ultimate: 2943.58375635
1000-Piece Puzzle: 2937.15736041
Overwatch 2: 2899.54822335
The Witcher 3: Wild Hunt: 2896.83756345
Fall Guys: 2892.72588832
Animal Crossing: New Horizons: 2884.69543147
Pillars of Eternity II: Deadfire: 2860.89340102


## 5- Random

Se realiza el procedimiento para recomendar a cada usuario 10 videojuegos aleatorios que no haya visto. Se define una función que realiza la recomendación top N para cada usuario, utilizando una seed de random para que sea replicable:

In [98]:
import random

def get_top_n_random(trainset, n=10):
    random.seed(42)
    all_items = set(iid for iid in range(trainset.n_items))
    top_n = defaultdict(list)

    for uid in trainset.all_users():
        user_items = set(iid for (iid, _) in trainset.ur[uid])
        available_items = list(all_items - user_items)
        random_items = random.sample(available_items, min(n, len(available_items)))
        top_n[trainset.to_raw_uid(uid)] = [(trainset.to_raw_iid(iid)) for iid in random_items]

    return top_n

Se obtiene la lista de recomendación top 10 para cada usuario y se muestra como ejemplo el caso del con ID = 457:

In [99]:
top_n_random = get_top_n_random(trainset, n=10)
print(top_n_random["457"])

['39', '36', '18', '13', '40', '23', '7', '1', '28', '4']


Se muestran los títulos de los videojuegos recomendados al usuario ID = 457:

In [100]:
for item_id in top_n_random["457"]:
    print(info_videojuegos[int(item_id)])

The Witcher 3: Wild Hunt
The Elder Scrolls V: Skyrim
Kingdom Hearts III
Hades
Tomb Raider (2013)
Pillars of Eternity II: Deadfire
Cuphead
1000-Piece Puzzle
Sid Meier’s Civilization VI
Bioshock Infinite
